In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt

In [ ]:
LABELS = ["MF","SK","SJ"]

WIN_PATH  = Path("windows_val.csv")

AUDIO_DIR = Path("preds_unimodal/test/audio")
VIDEO_DIR = Path("preds_unimodal/test/video")

TEXT_PRED_PATH = Path("llama_sft_20260201_215114_val/preds_with_context_20260201_215114_val.csv")
TEXT_VARIANT_NAME = "text_with_context"

OUT_DIR = Path("fusion_20260201_215114")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("OUT_DIR:", OUT_DIR)


In [ ]:
windows = pd.read_csv(WIN_PATH)

if "video_id" in windows.columns:
    ID_COL = "video_id"
elif "file_id" in windows.columns:
    ID_COL = "file_id"
else:
    raise ValueError(f"Have: {windows.columns.tolist()}")

# window start/end in seconds
if "w_start" in windows.columns and "w_end" in windows.columns:
    WS_COL, WE_COL = "w_start", "w_end"
elif "start_s" in windows.columns and "end_s" in windows.columns:
    WS_COL, WE_COL = "start_s", "end_s"
else:
    raise ValueError(f"Have: {windows.columns.tolist()}")

y_cols = [f"y_{l}" for l in LABELS]
if not all(c in windows.columns for c in y_cols):
    alt = [l for l in LABELS]
    if all(c in windows.columns for c in alt):
        y_cols = alt
    else:
        raise ValueError(f"Missing label")

y = windows[y_cols].to_numpy().astype(int)

if "mask_any" in windows.columns:
    mask = windows["mask_any"].to_numpy().astype(int)
elif "present" in windows.columns:
    mask = windows["present"].to_numpy().astype(int)
elif "mask" in windows.columns:
    mask = windows["mask"].to_numpy().astype(int)
else:
    mask = np.ones(len(windows), dtype=int)


windows.head()


In [ ]:
def load_modality_outputs(mod_dir: Path, split="test"):
    probs = np.load(mod_dir / f"probs_{split}.npy")
    conf  = np.load(mod_dir / f"conf_{split}.npy")
    pres  = np.load(mod_dir / f"present_{split}.npy")
    return probs, conf, pres

A_PROB, A_CONF, A_PRES = load_modality_outputs(AUDIO_DIR, "test")
V_PROB, V_CONF, V_PRES = load_modality_outputs(VIDEO_DIR, "test")

assert A_PROB.shape[0] == len(windows) and A_PROB.shape[1] == 3, A_PROB.shape
assert V_PROB.shape == A_PROB.shape, (V_PROB.shape, A_PROB.shape)

assert A_PRES.shape[0] == len(windows), A_PRES.shape
assert V_PRES.shape[0] == len(windows), V_PRES.shape

mask_av = (mask == 1) & (A_PRES.astype(int) == 1) & (V_PRES.astype(int) == 1)
mask_av = mask_av.astype(int)

print("Mask used for audio/video eval:", int(mask_av.sum()))

In [ ]:
import pandas as pd
import numpy as np

text_df = pd.read_csv(TEXT_PRED_PATH).copy()

if "video_id" not in text_df.columns and "file_id" in text_df.columns:
    text_df = text_df.rename(columns={"file_id":"video_id"})

text_df["start_ms"] = text_df["start_ms"].astype(int)
text_df["end_ms"]   = text_df["end_ms"].astype(int)

for l in LABELS:
    c = f"conf_{l}"
    p = f"prob_{l}"
    if c not in text_df.columns:
        text_df[c] = (text_df[p] - 0.5).abs()

win = windows[[ID_COL, WS_COL, WE_COL]].copy()
win = win.rename(columns={ID_COL:"video_id", WS_COL:"w_start_s", WE_COL:"w_end_s"})
win["w_start_ms"] = (win["w_start_s"] * 1000).round().astype(int)
win["w_end_ms"]   = (win["w_end_s"]   * 1000).round().astype(int)

T_PROB = np.full((len(win), 3), 0.5, dtype=float)
T_CONF = np.zeros((len(win), 3), dtype=float)

spans_by_vid = {vid: df for vid, df in text_df.groupby("video_id")}

for i, r in win.iterrows():
    vid = r["video_id"]
    if vid not in spans_by_vid:
        continue
    w0, w1 = int(r["w_start_ms"]), int(r["w_end_ms"])
    cand = spans_by_vid[vid]

    s0 = cand["start_ms"].to_numpy()
    s1 = cand["end_ms"].to_numpy()
    ov = np.maximum(0, np.minimum(w1, s1) - np.maximum(w0, s0))

    j = int(np.argmax(ov))
    if ov[j] <= 0:
        continue

    row = cand.iloc[j]
    T_PROB[i, :] = [row["prob_MF"], row["prob_SK"], row["prob_SJ"]]
    T_CONF[i, :] = [row["conf_MF"], row["conf_SK"], row["conf_SJ"]]

T_PRES = (T_CONF.sum(axis=1) > 0).astype(int)



In [ ]:
PLOT_DIR = OUT_DIR / "plots"
PLOT_DIR.mkdir(parents=True, exist_ok=True)

def prf(y_t, y_p):
    tp = int(((y_t==1)&(y_p==1)).sum())
    fp = int(((y_t==0)&(y_p==1)).sum())
    fn = int(((y_t==1)&(y_p==0)).sum())
    precision = tp/(tp+fp) if (tp+fp) else 0.0
    recall = tp/(tp+fn) if (tp+fn) else 0.0
    f1 = (2*precision*recall/(precision+recall)) if (precision+recall) else 0.0
    return dict(tp=tp, fp=fp, fn=fn, precision=precision, recall=recall, f1=f1)

def eval_probs(probs, mask_eval):
    mask_eval = mask_eval.astype(int)
    yt = y[mask_eval==1]
    yp = (probs[mask_eval==1] >= 0.5).astype(int)

    per = {LABELS[i]: prf(yt[:,i], yp[:,i]) for i in range(3)}
    macro_f1 = float(np.mean([per[l]["f1"] for l in LABELS]))

    micro_tp = int(((yt==1)&(yp==1)).sum())
    micro_fp = int(((yt==0)&(yp==1)).sum())
    micro_fn = int(((yt==1)&(yp==0)).sum())
    micro_prec = micro_tp/(micro_tp+micro_fp) if (micro_tp+micro_fp) else 0.0
    micro_rec  = micro_tp/(micro_tp+micro_fn) if (micro_tp+micro_fn) else 0.0
    micro_f1 = (2*micro_prec*micro_rec/(micro_prec+micro_rec)) if (micro_prec+micro_rec) else 0.0

    return {
        "n_eval": int((mask_eval==1).sum()),
        "label_support": {l:int(yt[:,i].sum()) for i,l in enumerate(LABELS)},
        "per_class": per,
        "macro_f1": macro_f1,
        "micro_f1": micro_f1,
        "micro_precision": micro_prec,
        "micro_recall": micro_rec,
    }

def save_prf_plot(name, metrics_dict):
    labels = LABELS
    prec = [metrics_dict["per_class"][l]["precision"] for l in labels]
    rec  = [metrics_dict["per_class"][l]["recall"] for l in labels]
    f1   = [metrics_dict["per_class"][l]["f1"] for l in labels]

    x = np.arange(len(labels))
    w = 0.25
    plt.figure(figsize=(8,5))
    plt.bar(x-w, prec, w, label="Precision")
    plt.bar(x,   rec,  w, label="Recall")
    plt.bar(x+w, f1,   w, label="F1")
    plt.xticks(x, labels)
    plt.ylim(0,1)
    plt.title(f"{name} – PRF per class")
    plt.ylabel("Score")
    plt.legend()
    plt.tight_layout()
    plt.savefig(PLOT_DIR / f"{name}_prf.png", dpi=300)
    plt.close()

def save_confusions(name, probs, mask_eval):
    mask_eval = mask_eval.astype(int)
    yt = y[mask_eval==1]
    yp = (probs[mask_eval==1] >= 0.5).astype(int)

    for i,l in enumerate(LABELS):
        tn = int(((yt[:,i]==0)&(yp[:,i]==0)).sum())
        fp = int(((yt[:,i]==0)&(yp[:,i]==1)).sum())
        fn = int(((yt[:,i]==1)&(yp[:,i]==0)).sum())
        tp = int(((yt[:,i]==1)&(yp[:,i]==1)).sum())
        cm = np.array([[tn, fp],[fn,tp]])

        plt.figure(figsize=(4,4))
        plt.imshow(cm)
        plt.xticks([0,1], ["Pred 0","Pred 1"])
        plt.yticks([0,1], ["True 0","True 1"])
        for (r,c),v in np.ndenumerate(cm):
            plt.text(c,r,str(v),ha="center",va="center")
        plt.title(f"{name} – {l}")
        plt.tight_layout()
        plt.savefig(PLOT_DIR / f"{name}_confusion_{l}.png", dpi=300)
        plt.close()

In [ ]:
def broadcast_conf(conf_arr, n_labels=3):
    conf_arr = np.asarray(conf_arr)
    if conf_arr.ndim == 1:
        return np.repeat(conf_arr[:, None], n_labels, axis=1)
    return conf_arr

def conf_from_prob(prob_arr):
    return np.abs(prob_arr - 0.5)

def fuse_conf_weighted(probs_list, confs_list, base_weights=None, eps=1e-8):
    if base_weights is None:
        base_weights = [1.0]*len(probs_list)

    num = 0.0
    den = 0.0
    for p, c, w in zip(probs_list, confs_list, base_weights):
        c2 = broadcast_conf(c, 3)
        num = num + w * p * c2
        den = den + w * c2
    return num / (den + eps)

In [ ]:
RUN_ID = OUT_DIR.name

mask_common = mask_av.copy()
mask_text_eval = (mask_common == 1) & (T_PRES == 1)
mask_text_eval = mask_text_eval.astype(int)

m_audio = eval_probs(A_PROB, mask_common)
m_video = eval_probs(V_PROB, mask_common)
m_text  = eval_probs(T_PROB, mask_text_eval)

F_TA  = fuse_conf_weighted([T_PROB, A_PROB], [T_CONF, A_CONF], base_weights=[1.0, 1.0])
F_TV  = fuse_conf_weighted([T_PROB, V_PROB], [T_CONF, V_CONF], base_weights=[1.0, 1.0])
F_AV  = fuse_conf_weighted([A_PROB, V_PROB], [A_CONF, V_CONF], base_weights=[1.0, 1.0])
F_TAV = fuse_conf_weighted([T_PROB, A_PROB, V_PROB], [T_CONF, A_CONF, V_CONF], base_weights=[1.0, 1.0, 1.0])

m_TA  = eval_probs(F_TA,  mask_common)
m_TV  = eval_probs(F_TV,  mask_common)
m_AV  = eval_probs(F_AV,  mask_common)
m_TAV = eval_probs(F_TAV, mask_common)

summary = [
    {"name":"audio", "variant":"unimodal", **m_audio},
    {"name":"video", "variant":"unimodal", **m_video},
    {"name":TEXT_VARIANT_NAME, "variant":"unimodal", **m_text},
    {"name":"fused_text+audio", "variant":"fusion", **m_TA},
    {"name":"fused_text+video", "variant":"fusion", **m_TV},
    {"name":"fused_audio+video", "variant":"fusion", **m_AV},
    {"name":"fused_text+audio+video", "variant":"fusion", **m_TAV},
]

summary_path = OUT_DIR / f"fusion_summary_{RUN_ID}.csv"
pd.DataFrame([{
    "name":s["name"],
    "variant":s["variant"],
    "n_eval":s["n_eval"],
    "macro_f1":s["macro_f1"],
    "micro_f1":s["micro_f1"],
    "micro_precision":s["micro_precision"],
    "micro_recall":s["micro_recall"],
    "support_MF":s["label_support"]["MF"],
    "support_SK":s["label_support"]["SK"],
    "support_SJ":s["label_support"]["SJ"],
} for s in summary]).to_csv(summary_path, index=False)

metrics_path = OUT_DIR / f"fusion_metrics_{RUN_ID}.json"
metrics_path.write_text(json.dumps({"run_id":RUN_ID, "results":summary}, indent=2), encoding="utf-8")

print("Saved:", summary_path)
print("Saved:", metrics_path)

for name, mets, probs, msk in [
    ("audio", m_audio, A_PROB, mask_common),
    ("video", m_video, V_PROB, mask_common),
    (TEXT_VARIANT_NAME, m_text, T_PROB, mask_text_eval),
    ("fused_text+audio", m_TA, F_TA, mask_common),
    ("fused_text+video", m_TV, F_TV, mask_common),
    ("fused_audio+video", m_AV, F_AV, mask_common),
    ("fused_text+audio+video", m_TAV, F_TAV, mask_common),
]:
    save_prf_plot(name, mets)
    save_confusions(name, probs, msk)

print("Saved plots to:", PLOT_DIR)

FUSED_PROB = F_TAV
FUSED_CONF = conf_from_prob(FUSED_PROB).astype(np.float32)
FUSED_PRES = mask_common.astype(np.int8)

np.save(OUT_DIR / "fused_probs_test.npy", FUSED_PROB.astype(np.float32))
np.save(OUT_DIR / "fused_conf_test.npy",  FUSED_CONF)
np.save(OUT_DIR / "fused_present_test.npy", FUSED_PRES)

print("Saved fused files:")
print(" -", OUT_DIR / "fused_probs_test.npy")
print(" -", OUT_DIR / "fused_conf_test.npy")
print(" -", OUT_DIR / "fused_present_test.npy")

df = pd.read_csv(summary_path).sort_values(["variant","micro_f1"], ascending=[True, False])
df
